**Zadanie 14 – Pipeline ML z różnymi transformerami** *(challenge)*

Zaimplementuj: `LogTransformer` (log(x+1)), `OutlierRemover` (usuwa wartości > 3·std od średniej), `BinningTransformer` (dzieli na przedziały). Stwórz pipeline i przetestuj na `[1, 5, 10, 100, 2, 3, 8, 200, 4, 6]`.

In [1]:

import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


# 1. Transformacja logarytmiczna
class LogTransformer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X)
        return np.log1p(X)


# 2. Usuwanie wartości odstających
class OutlierRemover(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X)

        mask = np.all(
            np.abs(X - self.mean_) <= 3 * self.std_,
            axis=1
        )

        return X[mask]


# 3. Dzielenie na przedziały
class BinningTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, n_bins=3):
        self.n_bins = n_bins

    def fit(self, X, y=None):
        X = np.asarray(X)

        self.min_ = np.min(X, axis=0)
        self.max_ = np.max(X, axis=0)

        self.bins_ = [
            np.linspace(a, b, self.n_bins + 1)[1:-1]
            for a, b in zip(
                self.min_.ravel(),
                self.max_.ravel()
            )
        ]

        return self

    def transform(self, X):
        X = np.asarray(X)

        return np.column_stack([
            np.digitize(X[:, i], self.bins_[i])
            for i in range(X.shape[1])
        ])


# Dane testowe
X = np.array([1, 5, 10, 100, 2, 3, 8, 200, 4, 6])
X = X.reshape(-1, 1)


# Pipeline
pipeline = Pipeline([
    ('log', LogTransformer()),
    ('outlier', OutlierRemover()),
    ('binning', BinningTransformer(n_bins=3))
])


# Test
result = pipeline.fit_transform(X)

print("Dane oryginalne:")
print(X.ravel())

print("\nWynik pipeline:")
print(result.ravel())


Dane oryginalne:
[  1   5  10 100   2   3   8 200   4   6]

Wynik pipeline:
[0 0 1 2 0 0 0 2 0 0]
